In [1]:
import sys
print(sys.executable)

C:\Users\SD1-06\miniforge3\envs\finrl\python.exe


In [2]:
%pip install -e C:\code\FinRL --no-deps

Obtaining file:///C:/code/FinRL
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for finrl (pyproject.toml): started
  Building editable for finrl (pyproject.toml): finished with status 'done'
  Created wheel for finrl: filename=finrl-0.3.8-py3-none-any.whl size=10688 sha256=b6b8fa4e45a57ab7584b3cf3ca118ab0a44112a63679d4e73d631b35ad4ff416
  Stored in directory: C:\Users\SD1-06\AppData\Local\Temp\pip-ephem-wheel-cache-mexlz3eo\wheels\f0\25\fd\e378110c6b03e8bed1190a2c1694231d27269c2df7547fbce8
Successfully built finrl
  Attemptin

In [3]:
%pip install gymnasium

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install stable-baselines3

Note: you may need to restart the kernel to use updated packages.


In [5]:
import finrl

print(finrl.__file__)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


C:\code\FinRL\finrl\__init__.py


In [6]:
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv

print("FinRL import 성공")

FinRL import 성공


In [7]:
from pypfopt import EfficientFrontier
from pypfopt import expected_returns
from pypfopt import risk_models

print("PyPortfolioOpt import 성공")

PyPortfolioOpt import 성공


In [10]:
from pathlib import Path

In [12]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results"

print(PROJECT_ROOT)
print(RESULTS_DIR)

C:\code\korea-finrl-portfolio
C:\code\korea-finrl-portfolio\results


In [14]:
import pandas as pd

portfolio_raw_df = YahooDownloader(
    start_date=START_DATE,
    end_date=END_DATE,
    ticker_list=KOREA_STOCKS
).fetch_data()

portfolio_raw_df["date"] = pd.to_datetime(
    portfolio_raw_df["date"]
)

print(portfolio_raw_df.shape)
portfolio_raw_df.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (9800, 8)
(9800, 8)


Price,date,close,high,low,open,volume,tic,day
0,2018-01-02,69346.695312,69980.411849,68984.571577,69980.411849,2014838,000660.KS,1
1,2018-01-02,110675.742188,113266.813075,110305.589204,111416.048155,731763,005380.KS,1
2,2018-01-02,41160.617188,41467.183917,40966.996095,41451.048826,8474250,005930.KS,1
3,2018-01-02,171199.250000,171199.250000,168297.567797,168491.013277,467935,035420.KS,1
4,2018-01-02,43072.527344,43618.613269,42594.702159,43618.613269,576658,105560.KS,1


In [18]:
KOREA_STOCKS = [
    "005930.KS",  # 삼성전자
    "000660.KS",  # SK하이닉스
    "005380.KS",  # 현대차
    "035420.KS",  # NAVER
    "105560.KS",  # KB금융
]

START_DATE = "2018-01-01"
END_DATE = "2026-01-01"

TRAIN_END = pd.Timestamp("2024-01-01")
VAL_END = pd.Timestamp("2025-01-01")

TIME_WINDOW = 50
INITIAL_AMOUNT = 100_000_000
COMMISSION = 0.0025

In [19]:
date_counts = (
    portfolio_raw_df
    .groupby("date")["tic"]
    .nunique()
)

common_dates = date_counts[
    date_counts == len(KOREA_STOCKS)
].index

portfolio_raw_df = (
    portfolio_raw_df[
        portfolio_raw_df["date"].isin(common_dates)
    ]
    .sort_values(["date", "tic"])
    .reset_index(drop=True)
)

print(portfolio_raw_df.groupby("tic").size())


tic
000660.KS    1960
005380.KS    1960
005930.KS    1960
035420.KS    1960
105560.KS    1960
dtype: int64


In [20]:
prices = (
    portfolio_raw_df
    .pivot(
        index="date",
        columns="tic",
        values="close"
    )
    .sort_index()
)

print(prices.shape)
print(prices.isna().sum())

prices.head()

(1960, 5)
tic
000660.KS    0
005380.KS    0
005930.KS    0
035420.KS    0
105560.KS    0
dtype: int64


tic,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
date,,,,,
2018-01-02,69346.695312,110675.742188,41160.617188,171199.25000,43072.527344
2018-01-03,70342.531250,111416.039062,41644.667969,168491.03125,43072.527344
2018-01-04,69799.328125,108454.804688,41209.000000,172746.81250,43004.269531
2018-01-05,71791.023438,110305.593750,42048.031250,175648.50000,43755.136719
2018-01-08,70795.164062,111786.187500,41967.355469,183773.21875,45461.644531


In [21]:
train_prices = prices[
    prices.index < TRAIN_END
].copy()

print("Train start:", train_prices.index.min())
print("Train end:", train_prices.index.max())
print("Train shape:", train_prices.shape)

train_prices.tail()

Train start: 2018-01-02 00:00:00
Train end: 2023-12-28 00:00:00
Train shape: (1475, 5)


tic,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
date,,,,,
2023-12-21,137807.375000,173189.656250,71298.812500,212422.062500,47361.640625
2023-12-22,137905.468750,173451.140625,72154.375000,210950.328125,46826.484375
2023-12-26,138199.703125,173276.812500,72819.843750,211440.890625,46826.484375
2023-12-27,138003.156250,173015.328125,74501.867188,219604.671875,46648.097656
2023-12-28,139084.359375,177373.406250,74979.437500,220589.437500,48253.582031


In [22]:
mu = expected_returns.mean_historical_return( #expected return:기대수익률
    train_prices
)

S = risk_models.sample_cov( #covariance matrix: 공분산 행
    train_prices
)

print("Expected Return")
display(mu)

print("\nCovariance Matrix")
display(S)

Expected Return


tic
000660.KS    0.126352
005380.KS    0.083975
005930.KS    0.107973
035420.KS    0.044288
105560.KS    0.019609
dtype: float64


Covariance Matrix


tic,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
tic,,,,,
000660.KS,0.132302,0.034739,0.062785,0.040657,0.033380
005380.KS,0.034739,0.108190,0.032492,0.028495,0.035409
005930.KS,0.062785,0.032492,0.063890,0.031420,0.029727
035420.KS,0.040657,0.028495,0.031420,0.122495,0.021579
105560.KS,0.033380,0.035409,0.029727,0.021579,0.100484


In [23]:
ef = EfficientFrontier(
    mu,
    S,
    weight_bounds=(0, 1)
)

ef.max_sharpe(  #sharpe ratio가 최대가 되는 포트폴리오 찾
    risk_free_rate=0.0
)

weights = ef.clean_weights()

weights

OrderedDict([('000660.KS', 0.14866),
             ('005380.KS', 0.16717),
             ('005930.KS', 0.68418),
             ('035420.KS', 0.0),
             ('105560.KS', 0.0)])

In [24]:
print(
    "Weight sum:",
    sum(weights.values())
)

Weight sum: 1.00001


In [25]:
mvo_weights = pd.DataFrame({
    "Ticker": list(weights.keys()),
    "Weight": list(weights.values())
})

mvo_weights["Weight (%)"] = (
    mvo_weights["Weight"] * 100
)

mvo_weights

,Ticker,Weight,Weight (%)
0,000660.KS,0.14866,14.866
1,005380.KS,0.16717,16.717
2,005930.KS,0.68418,68.418
3,035420.KS,0.00000,0.000
4,105560.KS,0.00000,0.000


In [26]:
df_mvo_val = portfolio_raw_df[
    (portfolio_raw_df["date"] >= TRAIN_END)
    & (portfolio_raw_df["date"] < VAL_END)
][
    ["date", "tic", "close", "high", "low"]
].copy()

print(df_mvo_val.shape)
print(df_mvo_val["date"].min())
print(df_mvo_val["date"].max())

print(
    df_mvo_val
    .groupby("tic")
    .size()
)

(1220, 5)
2024-01-02 00:00:00
2024-12-30 00:00:00
tic
000660.KS    244
005380.KS    244
005930.KS    244
035420.KS    244
105560.KS    244
dtype: int64


In [27]:
environment_mvo_val = PortfolioOptimizationEnv(
    df_mvo_val,
    initial_amount=INITIAL_AMOUNT,
    comission_fee_pct=COMMISSION,
    time_window=TIME_WINDOW,
    features=["close", "high", "low"],
    normalize_df=None
)

print(
    "환경 종목 순서:",
    environment_mvo_val._tic_list
)

환경 종목 순서: ['000660.KS' '005380.KS' '005930.KS' '035420.KS' '105560.KS']


In [29]:
import numpy as np

mvo_stock_weights = np.array([
    weights[tic]
    for tic in environment_mvo_val._tic_list
], dtype=float)

# clean_weights 반올림 오차 제거
mvo_stock_weights = (
    mvo_stock_weights
    / mvo_stock_weights.sum()
)

print(
    "Stock weights:",
    mvo_stock_weights
)

print(
    "합계:",
    mvo_stock_weights.sum()
)

Stock weights: [0.14865851 0.16716833 0.68417316 0.         0.        ]
합계: 1.0


In [30]:
mvo_action = np.concatenate(
    (
        [0.0],
        mvo_stock_weights
    )
)

print(
    "MVO action:",
    mvo_action
)

print(
    "Action sum:",
    mvo_action.sum()
)

MVO action: [0.         0.14865851 0.16716833 0.68417316 0.         0.        ]
Action sum: 1.0


In [31]:
environment_mvo_val.reset()

done = False

while not done:

    _, _, done, _ = environment_mvo_val.step(
        mvo_action
    )

Initial portfolio value:100000000
Final portfolio value: 80707856.0
Final accumulative portfolio value: 0.8070785403251648
Maximum DrawDown: -0.37931781654431795
Sharpe ratio: -0.7259821811007252


In [32]:
mvo_val_final = (
    environment_mvo_val
    ._asset_memory["final"][-1]
)

mvo_val_return = (
    mvo_val_final
    / INITIAL_AMOUNT
    - 1
) * 100

print(
    "MVO Validation final:",
    mvo_val_final
)

print(
    "MVO Validation return:",
    mvo_val_return
)

MVO Validation final: 80707860.0
MVO Validation return: -19.292147


In [33]:
val_start = environment_mvo_val._date_memory[0]
val_end = environment_mvo_val._date_memory[-1]

print("Actual evaluation period:")
print(val_start, "~", val_end)

Actual evaluation period:
2024-03-14 00:00:00 ~ 2024-12-30 00:00:00


In [34]:
val_prices = prices.loc[
    (prices.index >= val_start)
    & (prices.index <= val_end)
]

individual_returns = (
    val_prices.iloc[-1]
    / val_prices.iloc[0]
    - 1
) * 100

individual_returns = (
    individual_returns
    .sort_values(ascending=False)
)

individual_returns

tic
105560.KS     8.828751
000660.KS     7.905419
035420.KS     6.477514
005380.KS   -13.751881
005930.KS   -26.851431
dtype: float64

In [35]:
environment_mvo_val_no_fee = PortfolioOptimizationEnv(
    df_mvo_val,
    initial_amount=INITIAL_AMOUNT,
    comission_fee_pct=0.0,
    time_window=TIME_WINDOW,
    features=["close", "high", "low"],
    normalize_df=None
)

environment_mvo_val_no_fee.reset()

done = False

while not done:
    _, _, done, _ = environment_mvo_val_no_fee.step(
        mvo_action
    )

Initial portfolio value:100000000
Final portfolio value: 81059144.0
Final accumulative portfolio value: 0.810591459274292
Maximum DrawDown: -0.3781160876898144
Sharpe ratio: -0.7080722612716145


In [36]:
mvo_val_no_fee_final = (
    environment_mvo_val_no_fee
    ._asset_memory["final"][-1]
)

mvo_val_no_fee_return = (
    mvo_val_no_fee_final
    / INITIAL_AMOUNT
    - 1
) * 100

print("MVO with fee:", mvo_val_return)
print("MVO without fee:", mvo_val_no_fee_return)
print(
    "Fee/rebalancing drag:",
    mvo_val_return - mvo_val_no_fee_return
)

MVO with fee: -19.292147
MVO without fee: -18.940853
Fee/rebalancing drag: -0.35129356


In [37]:
rolling_dates = prices.loc[
    (prices.index >= val_start)
    & (prices.index <= val_end)
].index

print(
    rolling_dates[0],
    "~",
    rolling_dates[-1]
)

print(
    "평가 거래일:",
    len(rolling_dates)
)

2024-03-14 00:00:00 ~ 2024-12-30 00:00:00
평가 거래일: 195


In [38]:
LOOKBACK = 252

def calculate_mvo_weights(
    price_history,
    tickers
):
    mu = (
        expected_returns
        .mean_historical_return(
            price_history
        )
    )

    S = risk_models.sample_cov(
        price_history
    )

    ef = EfficientFrontier(
        mu,
        S,
        weight_bounds=(0, 1)
    )

    ef.max_sharpe(
        risk_free_rate=0.0
    )

    clean_weights = (
        ef.clean_weights()
    )

    stock_weights = np.array(
        [
            clean_weights[tic]
            for tic in tickers
        ],
        dtype=float
    )

    # 반올림 오차 보정
    stock_weights = (
        stock_weights
        / stock_weights.sum()
    )

    return stock_weights

In [39]:
rebalance_dates = (
    pd.Series(
        rolling_dates,
        index=rolling_dates
    )
    .groupby(
        rolling_dates.to_period("M")
    )
    .first()
    .tolist()
)

rebalance_dates

[Timestamp('2024-03-14 00:00:00'),
 Timestamp('2024-04-01 00:00:00'),
 Timestamp('2024-05-02 00:00:00'),
 Timestamp('2024-06-03 00:00:00'),
 Timestamp('2024-07-01 00:00:00'),
 Timestamp('2024-08-01 00:00:00'),
 Timestamp('2024-09-02 00:00:00'),
 Timestamp('2024-10-02 00:00:00'),
 Timestamp('2024-11-01 00:00:00'),
 Timestamp('2024-12-02 00:00:00')]

In [40]:
rolling_weight_history = {}

for rebalance_date in rebalance_dates:

    past_prices = prices[
        prices.index < rebalance_date
    ].tail(LOOKBACK)

    stock_weights = (
        calculate_mvo_weights(
            past_prices,
            environment_mvo_val._tic_list
        )
    )

    rolling_weight_history[
        rebalance_date
    ] = stock_weights

In [41]:
for date, weight in (
    list(
        rolling_weight_history.items()
    )[:3]
):

    print(date)
    print(weight)
    print()

2024-03-14 00:00:00
[0.44536 0.27903 0.      0.      0.27561]

2024-04-01 00:00:00
[0.66431664 0.16637166 0.00526005 0.         0.16405164]

2024-05-02 00:00:00
[0.60557 0.1544  0.      0.      0.24003]



In [42]:
#rolling mvo

environment_rolling_mvo_val = (
    PortfolioOptimizationEnv(
        df_mvo_val,
        initial_amount=INITIAL_AMOUNT,
        comission_fee_pct=COMMISSION,
        time_window=TIME_WINDOW,
        features=[
            "close",
            "high",
            "low"
        ],
        normalize_df=None
    )
)

environment_rolling_mvo_val.reset()

array([[[139969.   , 134464.6  , 134071.42 , 135152.64 , 133678.27 ,
         135054.36 , 131220.92 , 133678.27 , 131810.69 , 131810.69 ,
         129844.82 , 128763.61 , 133874.84 , 138887.77 , 140165.58 ,
         138396.33 , 139084.36 , 135054.36 , 133678.27 , 132695.31 ,
         134562.89 , 132400.42 , 130434.59 , 132597.   , 130041.42 ,
         135644.1  , 135644.1  , 140362.16 , 147439.23 , 146161.44 ,
         146358.03 , 144293.89 , 148717.05 , 147046.08 , 146456.33 ,
         153828.28 , 158644.6  , 159037.81 , 151174.36 , 155302.7  ,
         153533.42 , 163657.55 , 162969.52 , 160119.02 , 162084.86 ,
         168965.36 , 163755.86 , 162871.23 , 160807.08 , 159136.1  ],
        [174758.58 , 168918.77 , 165606.64 , 163253.27 , 161858.69 ,
         161771.52 , 162730.27 , 162991.77 , 162120.17 , 163427.58 ,
         162294.48 , 158459.36 , 157064.78 , 158372.2  , 156977.64 ,
         161074.22 , 161248.55 , 164473.52 , 163253.27 , 170487.64 ,
         165606.64 , 169616.02 , 

In [43]:
current_weights = None
done = False

while not done:

    current_date = (
        environment_rolling_mvo_val
        ._date_memory[-1]
    )

    if current_date in rolling_weight_history:

        current_weights = (
            rolling_weight_history[
                current_date
            ]
        )

    # 첫 평가일이면 반드시 비중 존재
    action = np.concatenate(
        (
            [0.0],
            current_weights
        )
    )

    _, _, done, _ = (
        environment_rolling_mvo_val
        .step(action)
    )

Initial portfolio value:100000000
Final portfolio value: 92945704.0
Final accumulative portfolio value: 0.9294570684432983
Maximum DrawDown: -0.2624023783744478
Sharpe ratio: -0.07116279563806635


In [44]:
rolling_mvo_val_final = (
    environment_rolling_mvo_val
    ._asset_memory["final"][-1]
)

rolling_mvo_val_return = (
    rolling_mvo_val_final
    / INITIAL_AMOUNT
    - 1
) * 100

print(
    "Static MVO:",
    mvo_val_return
)

print(
    "Rolling MVO:",
    rolling_mvo_val_return
)

print(
    "EIIE 40ep:",
    6.315410
)

Static MVO: -19.292147
Rolling MVO: -7.054293
EIIE 40ep: 6.31541


In [45]:
#2025 data

df_mvo_test = portfolio_raw_df[
    portfolio_raw_df["date"] >= VAL_END
][
    ["date", "tic", "close", "high", "low"]
].copy()

print(df_mvo_test.shape)
print(df_mvo_test["date"].min())
print(df_mvo_test["date"].max())

print(
    df_mvo_test
    .groupby("tic")
    .size()
)

(1205, 5)
2025-01-02 00:00:00
2025-12-30 00:00:00
tic
000660.KS    241
005380.KS    241
005930.KS    241
035420.KS    241
105560.KS    241
dtype: int64


In [46]:
# static mvo 2025 test

environment_mvo_test = PortfolioOptimizationEnv(
    df_mvo_test,
    initial_amount=INITIAL_AMOUNT,
    comission_fee_pct=COMMISSION,
    time_window=TIME_WINDOW,
    features=["close", "high", "low"],
    normalize_df=None
)

environment_mvo_test.reset()

done = False

while not done:
    _, _, done, _ = environment_mvo_test.step(
        mvo_action
    )

Initial portfolio value:100000000
Final portfolio value: 213576176.0
Final accumulative portfolio value: 2.1357617378234863
Maximum DrawDown: -0.1578012698903246
Sharpe ratio: 3.157858809278911


In [47]:
# 결과

static_mvo_test_final = (
    environment_mvo_test
    ._asset_memory["final"][-1]
)

static_mvo_test_return = (
    static_mvo_test_final
    / INITIAL_AMOUNT
    - 1
) * 100

print(
    "Static MVO Test final:",
    static_mvo_test_final
)

print(
    "Static MVO Test return:",
    static_mvo_test_return
)

Static MVO Test final: 213576180.0
Static MVO Test return: 113.57617


In [48]:
#rolling mvo 2025

test_start = (
    environment_mvo_test
    ._date_memory[0]
)

test_end = (
    environment_mvo_test
    ._date_memory[-1]
)

print(
    "Test evaluation period:",
    test_start,
    "~",
    test_end
)

Test evaluation period: 2025-03-19 00:00:00 ~ 2025-12-30 00:00:00


In [49]:
#날짜 기준으로 월별 rebalacing

rolling_test_dates = prices.loc[
    (prices.index >= test_start)
    & (prices.index <= test_end)
].index

rebalance_test_dates = (
    pd.Series(
        rolling_test_dates,
        index=rolling_test_dates
    )
    .groupby(
        rolling_test_dates.to_period("M")
    )
    .first()
    .tolist()
)

rebalance_test_dates

[Timestamp('2025-03-19 00:00:00'),
 Timestamp('2025-04-01 00:00:00'),
 Timestamp('2025-05-02 00:00:00'),
 Timestamp('2025-06-02 00:00:00'),
 Timestamp('2025-07-01 00:00:00'),
 Timestamp('2025-08-01 00:00:00'),
 Timestamp('2025-09-01 00:00:00'),
 Timestamp('2025-10-01 00:00:00'),
 Timestamp('2025-11-03 00:00:00'),
 Timestamp('2025-12-01 00:00:00')]

In [50]:
#그 날짜보다 과거 252 거래일만 사용해서 비중 계산

rolling_test_weight_history = {}

for rebalance_date in rebalance_test_dates:

    past_prices = prices[
        prices.index < rebalance_date
    ].tail(LOOKBACK)

    stock_weights = (
        calculate_mvo_weights(
            past_prices,
            environment_mvo_test._tic_list
        )
    )

    rolling_test_weight_history[
        rebalance_date
    ] = stock_weights

In [51]:
#확인

for date, weight in list(
    rolling_test_weight_history.items()
)[:3]:

    print(date)
    print(weight)
    print()

2025-03-19 00:00:00
[0.27729 0.      0.      0.26232 0.46039]

2025-04-01 00:00:00
[0.60136 0.      0.      0.07891 0.31973]

2025-05-02 00:00:00
[0.      0.      0.      0.15211 0.84789]



In [52]:
environment_rolling_mvo_test = PortfolioOptimizationEnv(
    df_mvo_test,
    initial_amount=INITIAL_AMOUNT,
    comission_fee_pct=COMMISSION,
    time_window=TIME_WINDOW,
    features=["close", "high", "low"],
    normalize_df=None
)

environment_rolling_mvo_test.reset()

current_weights = None
done = False

while not done:

    current_date = (
        environment_rolling_mvo_test
        ._date_memory[-1]
    )

    if current_date in rolling_test_weight_history:
        current_weights = (
            rolling_test_weight_history[
                current_date
            ]
        )

    action = np.concatenate(
        (
            [0.0],
            current_weights
        )
    )

    _, _, done, _ = (
        environment_rolling_mvo_test
        .step(action)
    )

Initial portfolio value:100000000
Final portfolio value: 169947984.0
Final accumulative portfolio value: 1.6994798183441162
Maximum DrawDown: -0.18373760415135176
Sharpe ratio: 2.0216322449327504


In [54]:
EIIE_40EP_TEST_RETURN = 20.541298
NAVER_ONLY_TEST_RETURN = 16.586517

In [55]:
print(
    "Static MVO:",
    static_mvo_test_return
)

print(
    "Rolling MVO:",
    rolling_mvo_test_return
)

print(
    "EIIE 40ep:",
    EIIE_40EP_TEST_RETURN
)

print(
    "NAVER Only:",
    NAVER_ONLY_TEST_RETURN
)

Static MVO: 113.57617
Rolling MVO: 69.94798
EIIE 40ep: 20.541298
NAVER Only: 16.586517


In [56]:
#2025 개별 종목 실제 수익률 확인

test_prices = prices.loc[
    (prices.index >= test_start)
    & (prices.index <= test_end)
]

individual_test_returns = (
    test_prices.iloc[-1]
    / test_prices.iloc[0]
    - 1
) * 100

individual_test_returns = (
    individual_test_returns
    .sort_values(ascending=False)
)

individual_test_returns

tic
000660.KS    218.046501
005930.KS    109.377992
105560.KS     57.405273
005380.KS     50.456155
035420.KS     16.586536
dtype: float64

In [57]:
# finrl없이 직접 static mvo 수익률 계산

daily_returns = (
    test_prices
    .pct_change()
    .dropna()
)

weight_series = pd.Series(
    mvo_stock_weights,
    index=environment_mvo_test._tic_list
)

direct_mvo_daily_returns = (
    daily_returns
    .dot(weight_series)
)

direct_mvo_curve = (
    1 + direct_mvo_daily_returns
).cumprod()

direct_mvo_return = (
    direct_mvo_curve.iloc[-1] - 1
) * 100

print(
    "Direct daily-rebalanced MVO:",
    direct_mvo_return
)

Direct daily-rebalanced MVO: 114.44248052438928


In [58]:
#거래비용 0일때

environment_mvo_test_no_fee = PortfolioOptimizationEnv(
    df_mvo_test,
    initial_amount=INITIAL_AMOUNT,
    comission_fee_pct=0.0,
    time_window=TIME_WINDOW,
    features=["close", "high", "low"],
    normalize_df=None
)

environment_mvo_test_no_fee.reset()

done = False

while not done:
    _, _, done, _ = (
        environment_mvo_test_no_fee.step(
            mvo_action
        )
    )

Initial portfolio value:100000000
Final portfolio value: 214442848.0
Final accumulative portfolio value: 2.1444284915924072
Maximum DrawDown: -0.15764869673438353
Sharpe ratio: 3.1738295857338574


In [59]:
mvo_test_no_fee_final = (
    environment_mvo_test_no_fee
    ._asset_memory["final"][-1]
)

mvo_test_no_fee_return = (
    mvo_test_no_fee_final
    / INITIAL_AMOUNT
    - 1
) * 100

print(
    "FinRL MVO without fee:",
    mvo_test_no_fee_return
)

print(
    "FinRL MVO with fee:",
    static_mvo_test_return
)

print(
    "Direct MVO:",
    direct_mvo_return
)

FinRL MVO without fee: 114.44285
FinRL MVO with fee: 113.57617
Direct MVO: 114.44248052438928


In [60]:
environment_equal_test = PortfolioOptimizationEnv(
    df_mvo_test,
    initial_amount=INITIAL_AMOUNT,
    comission_fee_pct=COMMISSION,
    time_window=TIME_WINDOW,
    features=["close", "high", "low"],
    normalize_df=None
)

environment_equal_test.reset()

equal_stock_weight = 1 / len(KOREA_STOCKS)

equal_action = np.array(
    [0.0]
    + [equal_stock_weight] * len(KOREA_STOCKS)
)

print("Equal Weight action:", equal_action)
print("합계:", equal_action.sum())

Equal Weight action: [0.  0.2 0.2 0.2 0.2 0.2]
합계: 1.0


In [61]:
#실행

done = False

while not done:
    _, _, done, _ = (
        environment_equal_test.step(
            equal_action
        )
    )

Initial portfolio value:100000000
Final portfolio value: 184718272.0
Final accumulative portfolio value: 1.8471827507019043
Maximum DrawDown: -0.16484592109278484
Sharpe ratio: 3.057286870518973


In [62]:
#결과

equal_test_final = (
    environment_equal_test
    ._asset_memory["final"][-1]
)

equal_test_return = (
    equal_test_final
    / INITIAL_AMOUNT
    - 1
) * 100

print(
    "Equal Weight Test final:",
    equal_test_final
)

print(
    "Equal Weight Test return:",
    equal_test_return
)

Equal Weight Test final: 184718270.0
Equal Weight Test return: 84.71828


In [63]:
#2024 validation 결과 저장

validation_summary = pd.DataFrame({
    "Strategy": [
        "Static MVO",
        "Rolling MVO",
        "Equal Weight",
        "EIIE 40ep"
    ],
    "Return (%)": [
        -19.292147,
        -7.054293,
        -1.789981,
        6.315410
    ]
})

validation_summary.to_csv(
    RESULTS_DIR / "validation_strategy_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

validation_summary

,Strategy,Return (%)
0,Static MVO,-19.292147
1,Rolling MVO,-7.054293
2,Equal Weight,-1.789981
3,EIIE 40ep,6.315410


In [64]:
#2025 test 전체 비교표 저장
#EIIE: ensemble of identical independent evaluators + RL
#시장 데이터를 관찰하면서 포트폴리오 비중을 직접 학습

test_strategy_summary = pd.DataFrame({
    "Strategy": [
        "Static MVO",
        "Equal Weight",
        "Rolling MVO",
        "EIIE 40ep",
        "NAVER Only"
    ],
    "Return (%)": [
        static_mvo_test_return,
        equal_test_return,
        rolling_mvo_test_return,
        20.541298,
        16.586517
    ],
    "MDD (%)": [
        -15.780127,
        -16.484592,
        -18.373760,
        -22.302587,
        -26.161767
    ],
    "Sharpe": [
        3.157859,
        3.057287,
        2.021632,
        0.808454,
        0.668188
    ]
})

test_strategy_summary.to_csv(
    RESULTS_DIR / "test_2025_strategy_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

test_strategy_summary

,Strategy,Return (%),MDD (%),Sharpe
0,Static MVO,113.576172,-15.780127,3.157859
1,Equal Weight,84.718277,-16.484592,3.057287
2,Rolling MVO,69.947983,-18.373760,2.021632
3,EIIE 40ep,20.541298,-22.302587,0.808454
4,NAVER Only,16.586517,-26.161767,0.668188


In [66]:
# mvo/ew 자산곡선 저장
# ew: equal weight 동일가중
# mvo: mean-variance optimization 평균-분산 최적

static_mvo_curve = pd.Series(
    environment_mvo_test._asset_memory["final"],
    index=pd.to_datetime(
        environment_mvo_test._date_memory
    ),
    name="Static MVO"
)

rolling_mvo_curve = pd.Series(
    environment_rolling_mvo_test._asset_memory["final"],
    index=pd.to_datetime(
        environment_rolling_mvo_test._date_memory
    ),
    name="Rolling MVO"
)

equal_weight_curve = pd.Series(
    environment_equal_test._asset_memory["final"],
    index=pd.to_datetime(
        environment_equal_test._date_memory
    ),
    name="Equal Weight"
)

mvo_test_curves = pd.concat(
    [
        static_mvo_curve,
        rolling_mvo_curve,
        equal_weight_curve
    ],
    axis=1
)

mvo_test_curves.to_csv(
    RESULTS_DIR / "test_2025_mvo_curves.csv",
    encoding="utf-8-sig"
)

mvo_test_curves.head()

,Static MVO,Rolling MVO,Equal Weight
2025-03-19,100000000.0,100000000.0,100000000.0
2025-03-20,102190800.0,101230896.0,101143032.0
2025-03-21,104539160.0,101589656.0,102323632.0
2025-03-24,103540736.0,101032784.0,102223168.0
2025-03-25,103031000.0,100687344.0,102389728.0
